# Projet Kayak — Partie 2 : Visualisation des destinations

On reprend `df_weather` (sauvegardé en CSV à la fin de la partie 1) et on construit deux cartes interactives avec **Plotly** pour identifier visuellement les meilleures destinations de la semaine.

## 1. Imports et rechargement des données

In [2]:
import pandas as pd
import plotly.express as px

# On recharge depuis le CSV pour pouvoir relancer cette partie sans rappeler les APIs
df_weather = pd.read_csv("data/weather.csv")
df_weather

,city,lat,lon,temp_avg,rain_total,clouds_avg,weather_score
0,Mont Saint Michel,48.635954,-1.511460,17.3,0.0,47.0,12.60
1,Saint Malo,48.649518,-2.026041,17.1,0.0,54.0,11.70
2,Bayeux,49.276462,-0.702474,18.0,0.0,42.9,13.71
3,Le Havre,49.493898,0.107973,17.8,0.0,41.3,13.67
4,Rouen,49.440459,1.093966,19.7,0.0,42.6,15.44
5,Paris,48.858890,2.320041,23.3,1.0,37.6,19.04
6,Amiens,49.894171,2.295695,18.7,0.0,42.2,14.48
7,Lille,50.636565,3.063528,19.2,0.2,43.5,14.75
8,Strasbourg,48.584614,7.750713,24.0,11.7,49.4,13.21
9,Chateau du Haut Koenigsbourg,48.249382,7.343941,20.7,16.1,41.5,8.50


In [3]:
import plotly 
print(plotly.__version__)

5.24.1


## 2. Carte 1 — Score météo des 35 villes

On affiche **toutes les villes** sur une carte de France. Chaque ville est un point coloré selon son score météo : **rouge = score élevé** (destination attractive), **bleu = score faible** (à éviter cette semaine).

Plotly Express permet de produire ce type de carte en quelques lignes avec `px.scatter_map`. Pas besoin de clé Mapbox : le style `open-street-map` est gratuit et libre.

In [4]:
# On crée une colonne "marker_size" pour la taille des marqueurs.
# Astuce : on décale tous les scores pour les rendre positifs (le minimum devient 1),
# tout en conservant la colonne weather_score d'origine pour la couleur et le hover.
df_weather["marker_size"] = df_weather["weather_score"] - df_weather["weather_score"].min() + 1

fig1 = px.scatter_map(
    df_weather,
    lat="lat",
    lon="lon",
    color="weather_score",                # couleur basée sur le vrai score (négatifs OK ici)
    size="marker_size",                   # taille basée sur la colonne décalée (toujours > 0)
    size_max=20,
    hover_name="city",
    hover_data={
        "temp_avg": ":.1f",
        "rain_total": ":.1f",
        "clouds_avg": ":.0f",
        "weather_score": ":.2f",          # on affiche le vrai score dans le tooltip
        "marker_size": False,             # on masque la colonne technique
        "lat": False,
        "lon": False,
    },
    color_continuous_scale="RdYlBu_r",
    zoom=4.5,
    center={"lat": 46.6, "lon": 2.5},
    height=600,
    title="Score météo des 35 destinations — prévisions à 5 jours",
)

# fig1.update_layout(map_style="open-street-map")        #  provoque une erreur 403r exigeant un en-tête Referer valide que VSCode n'envoie pas
fig1.update_layout(map_style="carto-positron")  
fig1.show()

> **Note** : si ta version de Plotly est antérieure à 5.24, remplace `px.scatter_map` par `px.scatter_mapbox` et `map_style` par `mapbox_style` — le reste du code est identique.

## 3. Carte 2 — Top 5 des destinations

On extrait les 5 meilleures villes selon le score météo et on les met en valeur sur une carte dédiée. C'est ce top 5 qui servira ensuite à orienter le scraping Booking (on ne scrapera les hôtels que pour ces villes-là, pour économiser temps et bande passante).

In [5]:
top5 = df_weather.nlargest(5, "weather_score").reset_index(drop=True)
top5

,city,lat,lon,temp_avg,rain_total,clouds_avg,weather_score,marker_size
0,Marseille,43.296399,5.377789,28.8,0.0,11.2,27.68,20.90
1,Aix en Provence,43.529842,5.447474,29.7,0.1,20.6,27.59,20.81
2,Avignon,43.949249,4.805901,29.8,1.4,15.4,27.56,20.78
3,Nimes,43.837425,4.360069,30.0,1.8,15.6,27.54,20.76
4,Aigues Mortes,43.566152,4.191540,29.0,0.2,17.2,27.18,20.40


In [6]:
fig2 = px.scatter_map(
    top5,
    lat="lat",
    lon="lon",
    color="weather_score",
    size="weather_score",
    size_max=30,                          # plus gros que carte 1 pour bien voir le top 5
    hover_name="city",
    hover_data={
        "temp_avg": ":.1f",
        "rain_total": ":.1f",
        "lat": False,
        "lon": False,
    },
    text="city",                          # le nom de la ville s'affiche directement à côté du point
    color_continuous_scale="RdYlBu_r",
    zoom=4.5,
    center={"lat": 46.6, "lon": 2.5},
    height=600,
    title="Top 5 des destinations météo — semaine à venir",
)

fig2.update_layout(map_style="carto-positron")  
fig2.update_traces(textposition="top right")  # placement du texte à côté des points

fig2.show()

## 4. Sauvegarde du top 5

On sauvegarde la liste des 5 villes : c'est elle qui pilotera le scraping Booking dans la partie suivante.

In [7]:
top5.to_csv("data/top5_cities.csv", index=False)
print("✅ Top 5 sauvegardé dans data/top5_cities.csv")
print("\nVilles à scraper sur Booking :")
for city in top5["city"]:
    print(f"  - {city}")

✅ Top 5 sauvegardé dans data/top5_cities.csv

Villes à scraper sur Booking :
  - Marseille
  - Aix en Provence
  - Avignon
  - Nimes
  - Aigues Mortes
